# __Preprocessing__

In [113]:
import pandas as pd
import os

print(os.getcwd())

data = pd.read_csv('../data/raw/datos_tdah.csv')
data.head()

/Users/juan/Desktop/THAD/TDAH-data/notebooks


,CÓDIGO,EDAD,SEXO,ESCOLARIDAD,LATERALIDAD,ANTECEDENTES FAMILIARES,ANTECEDENTES OBSTETRICOS,RETRASOS EN EL DESARROLLO,ETIQUETA,INDICE TDAH,INIDICE INATENCIÓN,INDICE IMPULS/HIPERAC,CIT,Agresividad BASC-3,Hiperactividad BASC-3,Problemas de conducta BASC-3,Problemas de atención BASC-3,Atipicidad BASC-3,F.FONOLOGICA
0,4,12,1,6,1,3,0,3,TDAH INATENTO,0.39,0.67,0.11,89,42,50,46,73,51,18
1,9,9,2,4,1,2,1,3,TDAH INATENTO,0.39,0.67,0.11,88,47,50,46,76,58,14
2,11,11,1,5,1,1,0,3,TDAH INATENTO,0.78,1.00,0.56,103,38,53,43,70,54,11
3,12,10,1,5,1,1,0,3,TDAH INATENTO,0.78,1.00,0.56,103,53,43,43,72,51,11
4,33,6,1,1,1,4,0,0,TDAH INATENTO,0.61,0.89,0.33,93,49,38,38,72,42,10


## __Verificación de inconsistencias en el cálculo del índice TDAH__

Se realizó una validación del campo `INDICE TDAH` comparándolo con el promedio esperado de los dos componentes clínicos reportados:


$\text{INDICE TDAH} = \frac{\text{INDICE INATENCIÓN} + \text{INDICE IMPULS/HIPERAC}}{2}$


Las filas identificadas como inconsistentes presentan diferencias que no pueden explicarse únicamente por redondeo decimal. En estos casos, el valor almacenado en `INDICE TDAH` no corresponde al promedio esperado de las subescalas de inatención e hiperactividad/impulsividad.

Por esta razón, dichas filas fueron consideradas inconsistentes y se recomienda:
- corregir el valor del índice si se dispone de la fuente original, o
- excluir temporalmente estos registros de los análisis estadísticos y modelos predictivos para evitar sesgos o errores de interpretación.



In [114]:
codigos_inconsistentes = [
    395, 302, 15330, 15329, 854, 1363, 9785, 9790,
    9783, 5678, 5679, 5680, 5691, 5703, 782, 1320
]

data = data[~data["CÓDIGO"].isin(codigos_inconsistentes)].reset_index(drop=True)
data.head()

,CÓDIGO,EDAD,SEXO,ESCOLARIDAD,LATERALIDAD,ANTECEDENTES FAMILIARES,ANTECEDENTES OBSTETRICOS,RETRASOS EN EL DESARROLLO,ETIQUETA,INDICE TDAH,INIDICE INATENCIÓN,INDICE IMPULS/HIPERAC,CIT,Agresividad BASC-3,Hiperactividad BASC-3,Problemas de conducta BASC-3,Problemas de atención BASC-3,Atipicidad BASC-3,F.FONOLOGICA
0,4,12,1,6,1,3,0,3,TDAH INATENTO,0.39,0.67,0.11,89,42,50,46,73,51,18
1,9,9,2,4,1,2,1,3,TDAH INATENTO,0.39,0.67,0.11,88,47,50,46,76,58,14
2,11,11,1,5,1,1,0,3,TDAH INATENTO,0.78,1.00,0.56,103,38,53,43,70,54,11
3,12,10,1,5,1,1,0,3,TDAH INATENTO,0.78,1.00,0.56,103,53,43,43,72,51,11
4,33,6,1,1,1,4,0,0,TDAH INATENTO,0.61,0.89,0.33,93,49,38,38,72,42,10


In [115]:
data.columns = data.columns.str.strip()
data.columns

Index(['CÓDIGO', 'EDAD', 'SEXO', 'ESCOLARIDAD', 'LATERALIDAD',
       'ANTECEDENTES FAMILIARES', 'ANTECEDENTES OBSTETRICOS',
       'RETRASOS EN EL DESARROLLO', 'ETIQUETA', 'INDICE TDAH',
       'INIDICE INATENCIÓN', 'INDICE IMPULS/HIPERAC', 'CIT',
       'Agresividad BASC-3', 'Hiperactividad BASC-3',
       'Problemas de conducta BASC-3', 'Problemas de atención BASC-3',
       'Atipicidad BASC-3', 'F.FONOLOGICA'],
      dtype='str')

## __Eliminación de registro inconsistente en lateralidad__

Durante el proceso de depuración de datos se identificó un registro con valor `0` en la variable `lateralidad`. Este valor no correspondía a ninguna de las categorías definidas en el diccionario de datos (`Diestro`, `Zurdo` o `Ambidiestro`), por lo que fue considerado inconsistente. 

Debido a que solo se identificó un caso y no fue posible establecer una equivalencia válida, el registro fue eliminado antes del análisis exploratorio de datos (EDA).

In [116]:
# Eliminar registros donde lateralidad sea 0
data = data[data["LATERALIDAD"] != 0].copy()

# Reiniciar índices
data = data.reset_index(drop=True)

# Verificar
print(data["LATERALIDAD"].value_counts(dropna=False))


LATERALIDAD
1    800
2     84
3      7
Name: count, dtype: int64


In [117]:
import json

# Abrir archivo JSON
with open("../data/raw/equivalencias.json", "r", encoding="utf-8") as f:
    diccionario = json.load(f)

# Columnas que deseas transformar
columnas_a_reemplazar = [
    "LATERALIDAD",
    "ANTECEDENTES FAMILIARES",
    "RETRASOS EN EL DESARROLLO",
    "ANTECEDENTES OBSTETRICOS"
]

In [118]:
for columna in columnas_a_reemplazar:
    if columna in data.columns and columna in diccionario:
        mapa = {}
        for item in diccionario[columna]:
            codigo = int(item["codigo"])
            etiqueta = item["etiqueta"]
            mapa[codigo] = etiqueta
        data[columna] = data[columna].replace(mapa)

data.head()    

,CÓDIGO,EDAD,SEXO,ESCOLARIDAD,LATERALIDAD,ANTECEDENTES FAMILIARES,ANTECEDENTES OBSTETRICOS,RETRASOS EN EL DESARROLLO,ETIQUETA,INDICE TDAH,INIDICE INATENCIÓN,INDICE IMPULS/HIPERAC,CIT,Agresividad BASC-3,Hiperactividad BASC-3,Problemas de conducta BASC-3,Problemas de atención BASC-3,Atipicidad BASC-3,F.FONOLOGICA
0,4,12,1,6,Diestro,Otro trastorno psicopatológico,No hay,Personal/social,TDAH INATENTO,0.39,0.67,0.11,89,42,50,46,73,51,18
1,9,9,2,4,Diestro,Otro trastorno del neurodesarrollo,Embarazo,Personal/social,TDAH INATENTO,0.39,0.67,0.11,88,47,50,46,76,58,14
2,11,11,1,5,Diestro,Familiares con TDAH,No hay,Personal/social,TDAH INATENTO,0.78,1.00,0.56,103,38,53,43,70,54,11
3,12,10,1,5,Diestro,Familiares con TDAH,No hay,Personal/social,TDAH INATENTO,0.78,1.00,0.56,103,53,43,43,72,51,11
4,33,6,1,1,Diestro,Otros,No hay,No reporta,TDAH INATENTO,0.61,0.89,0.33,93,49,38,38,72,42,10


In [119]:
import pandas as pd
import unicodedata

# =========================
# 1. Limpiar nombres originales
# =========================

data = data.copy()

# Quitar espacios iniciales/finales
data.columns = data.columns.str.strip()

# =========================
# 2. Renombrar columnas
# =========================

renombrar_columnas = {
    "CÓDIGO": "codigo",
    "EDAD": "edad",
    "SEXO": "sexo",
    "ESCOLARIDAD": "escolaridad",
    "LATERALIDAD": "lateralidad",
    "ANTECEDENTES FAMILIARES": "antecedentes_familiares",
    "ANTECEDENTES OBSTETRICOS": "antecedentes_obstetricos",
    "RETRASOS EN EL DESARROLLO": "retrasos_desarrollo",
    "ETIQUETA": "etiqueta",
    "INDICE TDAH": "indice_tdah",
    "INIDICE INATENCIÓN": "indice_inatencion",
    "INDICE IMPULS/HIPERAC": "indice_impuls_hiperac",
    "CIT": "cit",
    "Agresividad BASC-3": "agresividad_basc3",
    "Hiperactividad BASC-3": "hiperactividad_basc3",
    "Problemas de conducta BASC-3": "problemas_conducta_basc3",
    "Problemas de atención BASC-3": "problemas_atencion_basc3",
    "Atipicidad BASC-3": "atipicidad_basc3",
    "F.FONOLOGICA": "fluidez_fonologica"
}

data = data.rename(columns=renombrar_columnas)

# =========================
# 3. Usar codigo como índice
# =========================

data = data.set_index("codigo")
data.index.name = "codigo"

# =========================
# 4. Reemplazar variables categóricas
# =========================

# Sexo: 1 = Hombre, 2 = Mujer
data["sexo"] = data["sexo"].replace({
    1: "Hombre",
    2: "Mujer"
})

# Estandarizar texto en variables categóricas ya recodificadas
columnas_categoricas = [
    "sexo",
    "lateralidad",
    "antecedentes_familiares",
    "antecedentes_obstetricos",
    "retrasos_desarrollo",
    "etiqueta"
]

for col in columnas_categoricas:
    data[col] = data[col].astype(str).str.strip()

# =========================
# 5. Estandarizar etiquetas clínicas
# =========================

mapa_etiquetas = {
    "TDAH INATENTO": "TDAH inatento",
    "TDAH inatento + TND": "TDAH inatento + TND",
    "TDAH INATENTO + TEAZ": "TDAH inatento + TEAZ",
    "TDAH INATENTO + TEA": "TDAH inatento + TEA",
    "TDAH Hipe/Impul": "TDAH hiperactivo/impulsivo",
    "TDAH Hipe/Impul + TND": "TDAH hiperactivo/impulsivo + TND",
    "TDAH Hipe/Impul + TEAZ": "TDAH hiperactivo/impulsivo + TEAZ",
    "TDAH Hipe/Impul + TEA": "TDAH hiperactivo/impulsivo + TEA",
    "TDAH COMBINADO": "TDAH combinado",
    "TDAH COMBINADO + TND": "TDAH combinado + TND",
    "TDAH COMBINADO + TEAZ": "TDAH combinado + TEAZ",
    "TDAH COMBINADO + TEA": "TDAH combinado + TEA",
    "DESARROLLO TIPICO": "Desarrollo típico"
}

data["etiqueta"] = data["etiqueta"].replace(mapa_etiquetas)

# =========================
# 6. Asegurar variables numéricas
# =========================

columnas_enteras = [
    "edad",
    "escolaridad",
    "cit",
    "agresividad_basc3",
    "hiperactividad_basc3",
    "problemas_conducta_basc3",
    "problemas_atencion_basc3",
    "atipicidad_basc3",
    "fluidez_fonologica"
]

columnas_decimales = [
    "indice_tdah",
    "indice_inatencion",
    "indice_impuls_hiperac"
]

for col in columnas_enteras:
    data[col] = pd.to_numeric(data[col], errors="coerce").astype("Int64")

for col in columnas_decimales:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# =========================
# 7. Ordenar columnas
# Etiqueta queda al final
# =========================

orden_columnas = [
    "edad",
    "sexo",
    "escolaridad",
    "lateralidad",
    "antecedentes_familiares",
    "antecedentes_obstetricos",
    "retrasos_desarrollo",
    "indice_tdah",
    "indice_inatencion",
    "indice_impuls_hiperac",
    "cit",
    "agresividad_basc3",
    "hiperactividad_basc3",
    "problemas_conducta_basc3",
    "problemas_atencion_basc3",
    "atipicidad_basc3",
    "fluidez_fonologica",
    "etiqueta"
]

data = data[orden_columnas]



In [120]:
data.head()

,edad,sexo,escolaridad,lateralidad,antecedentes_familiares,antecedentes_obstetricos,retrasos_desarrollo,indice_tdah,indice_inatencion,indice_impuls_hiperac,cit,agresividad_basc3,hiperactividad_basc3,problemas_conducta_basc3,problemas_atencion_basc3,atipicidad_basc3,fluidez_fonologica,etiqueta
codigo,,,,,,,,,,,,,,,,,,
4,12,Hombre,6,Diestro,Otro trastorno psicopatológico,No hay,Personal/social,0.39,0.67,0.11,89,42,50,46,73,51,18,TDAH inatento
9,9,Mujer,4,Diestro,Otro trastorno del neurodesarrollo,Embarazo,Personal/social,0.39,0.67,0.11,88,47,50,46,76,58,14,TDAH inatento
11,11,Hombre,5,Diestro,Familiares con TDAH,No hay,Personal/social,0.78,1.00,0.56,103,38,53,43,70,54,11,TDAH inatento
12,10,Hombre,5,Diestro,Familiares con TDAH,No hay,Personal/social,0.78,1.00,0.56,103,53,43,43,72,51,11,TDAH inatento
33,6,Hombre,1,Diestro,Otros,No hay,No reporta,0.61,0.89,0.33,93,49,38,38,72,42,10,TDAH inatento


In [121]:
data.info()

<class 'pandas.DataFrame'>
Index: 891 entries, 4 to 3331
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   edad                      891 non-null    Int64  
 1   sexo                      891 non-null    str    
 2   escolaridad               891 non-null    Int64  
 3   lateralidad               891 non-null    str    
 4   antecedentes_familiares   891 non-null    str    
 5   antecedentes_obstetricos  891 non-null    str    
 6   retrasos_desarrollo       891 non-null    str    
 7   indice_tdah               891 non-null    float64
 8   indice_inatencion         891 non-null    float64
 9   indice_impuls_hiperac     891 non-null    float64
 10  cit                       891 non-null    Int64  
 11  agresividad_basc3         891 non-null    Int64  
 12  hiperactividad_basc3      891 non-null    Int64  
 13  problemas_conducta_basc3  891 non-null    Int64  
 14  problemas_atencion_basc3 

In [122]:
import pandas as pd


esperado = ((data["indice_inatencion"] + data["indice_impuls_hiperac"]) / 2).round(2)
diferencia = (data["indice_tdah"] - esperado).abs().round(2)

inconsistentes = data[diferencia > 0.02].copy()
inconsistentes["indice_tdah_esperado"] = esperado[diferencia > 0.02]
inconsistentes["diferencia"] = diferencia[diferencia > 0.02]

data = data[diferencia <= 0.02].reset_index(drop=True)

print("Filas inconsistentes:", len(inconsistentes))
inconsistentes.index


Filas inconsistentes: 16


Index(['395', '302', '15330', '15329', '854', '1363', '9785', '9790', '9783',
       '5678', '5679', '5680', '5691', '5703', '782', '1320'],
      dtype='str', name='codigo')

In [123]:
# Eliminar columna codigo
if "codigo" in data.columns:
    data = data.drop(columns=["codigo"])

# Reiniciar índices
data = data.reset_index(drop=True)

# Verificar
data.head()

,edad,sexo,escolaridad,lateralidad,antecedentes_familiares,antecedentes_obstetricos,retrasos_desarrollo,indice_tdah,indice_inatencion,indice_impuls_hiperac,cit,agresividad_basc3,hiperactividad_basc3,problemas_conducta_basc3,problemas_atencion_basc3,atipicidad_basc3,fluidez_fonologica,etiqueta
0,12,Hombre,6,Diestro,Otro trastorno psicopatológico,No hay,Personal/social,0.39,0.67,0.11,89,42,50,46,73,51,18,TDAH inatento
1,9,Mujer,4,Diestro,Otro trastorno del neurodesarrollo,Embarazo,Personal/social,0.39,0.67,0.11,88,47,50,46,76,58,14,TDAH inatento
2,11,Hombre,5,Diestro,Familiares con TDAH,No hay,Personal/social,0.78,1.00,0.56,103,38,53,43,70,54,11,TDAH inatento
3,10,Hombre,5,Diestro,Familiares con TDAH,No hay,Personal/social,0.78,1.00,0.56,103,53,43,43,72,51,11,TDAH inatento
4,6,Hombre,1,Diestro,Otros,No hay,No reporta,0.61,0.89,0.33,93,49,38,38,72,42,10,TDAH inatento


## __Versión Final__

In [124]:
# Guardar DataFrame procesado

ruta_salida = "../data/processed/processed_data.csv"

data.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo guardado en: {ruta_salida}")

Archivo guardado en: ../data/processed/processed_data.csv
